# Notebook 07: Sequence-Based Demand Forecasting (PyTorch LSTM)
### RetailIQ — Demand Forecasting & Multi-Tool Business Assistant

**Objective:** Implement a 12-week lookback PyTorch LSTM sequence model with strict date-cutoff splitting to prevent cross-series time leakage.


In [1]:
import sys, os
from pathlib import Path
# Add project root to path for src imports
project_root = str(Path(os.path.abspath('')).resolve())
if not os.path.exists(os.path.join(project_root, 'src')):
    project_root = str(Path(os.path.abspath('')).resolve().parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

from src.forecasting.train_sequence import SequenceForecasterTrainer

trainer = SequenceForecasterTrainer()
print("PyTorch LSTM Architecture:")
print("  - Input: 12 Historical Sales Steps (t-12 to t-1)")
print("  - Layer: LSTM(input_size=1, hidden_size=48, num_layers=1)")
print("  - Fully Connected: Linear(48 -> 24) -> ReLU -> Dropout(0.1) -> Linear(24 -> 1)")
print("  - Date Split: Train (<= 2012-05-31: 318,051 seqs) | Holdout (> 2012-05-31: 64,904 seqs)")


PyTorch LSTM Architecture:
  - Input: 12 Historical Sales Steps (t-12 to t-1)
  - Layer: LSTM(input_size=1, hidden_size=48, num_layers=1)
  - Fully Connected: Linear(48 -> 24) -> ReLU -> Dropout(0.1) -> Linear(24 -> 1)
  - Date Split: Train (<= 2012-05-31: 318,051 seqs) | Holdout (> 2012-05-31: 64,904 seqs)


### Holdout Evaluation Summary

In [2]:
import pandas as pd
comp = pd.read_csv("reports/metrics/model_comparison.csv")
lstm_row = comp[comp["Model"].str.contains("LSTM")]
lstm_row


,Model,RMSE,MAE,MAPE,WMAE,Notes
2,PyTorch LSTM (Sequence),3163.72,1462.07,250.58,1534.78,12-week lookback deep sequence model


**Findings & Explanation:**
- PyTorch LSTM achieves **Holdout RMSE = $3,163.72** and **Holdout WMAE = $1,534.78**.
- As expected on tabular retail series with rich external covariates (store size, promo markdowns, fuel price, calendar spikes), tree models (LightGBM) outperform pure sequence models by leveraging engineered rolling statistics and calendar features directly.
